In [1]:
# ===== Gradient Boosting (train on log target; evaluate raw + log) =====
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# 1) Load dataset
df = pd.read_csv("fixed_cleaned_crmls_dataset.txt")

In [3]:
df

,Unnamed: 0,ViewYN,PoolPrivateYN,ClosePrice,Latitude,Longitude,LivingArea,CountyOrParish,AttachedGarageYN,ParkingTotal,BathroomsTotalInteger,City,BedroomsTotal,FireplaceYN,LotSizeArea,NewConstructionYN,HighSchoolDistrict,PostalCode,BuildingAge,StoriesFinal
0,15,True,False,681877.0,33.725080,-117.222302,2824.0,Riverside,True,2.0,3.0,Menifee,5.0,False,7000.0,True,Mendocino Unified,92586,1.0,2.0
1,17,True,False,900000.0,34.203479,-118.643567,2500.0,Los Angeles,True,2.0,3.0,Los Angeles,5.0,True,8336.0,False,Call Listing Office,91307,54.0,2.0
2,19,False,False,862000.0,34.460368,-118.490755,2363.0,Los Angeles,True,2.0,3.0,Saugus,5.0,True,11705.0,False,William S. Hart Union,91390,30.0,2.0
3,20,False,False,5149000.0,34.043218,-118.519477,3338.0,Los Angeles,False,2.0,5.0,Pacific Palisades,4.0,True,6502.0,False,NaN,90272,21.0,2.0
4,25,False,False,1710000.0,37.669495,-121.763793,2485.0,Alameda,True,2.0,3.0,Livermore,4.0,True,5093.0,False,Livermore Valley,94550,14.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35772,125414,False,False,566990.0,33.700193,-117.371513,1435.0,Riverside,True,2.0,2.0,Lake Elsinore,3.0,False,6680.0,True,Lake Elsinore Unified,92530,1.0,1.0
35773,125420,True,False,495000.0,34.378787,-117.330930,2226.0,San Bernardino,True,2.0,3.0,Hesperia,4.0,True,18000.0,False,Hesperia Unified,92345,35.0,2.0
35774,125421,True,False,435000.0,34.854045,-119.148141,1753.0,Kern,True,2.0,3.0,Pine Mountain Club,3.0,True,10104.0,False,El Tejon Unified,93225,19.0,2.0
35775,125447,True,False,740000.0,33.891018,-118.087100,1186.0,Los Angeles,False,4.0,1.0,Norwalk,2.0,True,6211.0,False,Norwalk - La Mirada,90650,76.0,1.0


In [5]:
df.columns

Index(['Unnamed: 0', 'ViewYN', 'PoolPrivateYN', 'ClosePrice', 'Latitude',
       'Longitude', 'LivingArea', 'CountyOrParish', 'AttachedGarageYN',
       'ParkingTotal', 'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'FireplaceYN', 'LotSizeArea', 'NewConstructionYN', 'HighSchoolDistrict',
       'PostalCode', 'BuildingAge', 'StoriesFinal'],
      dtype='object')

In [9]:
# 2) Target / Features
y = df["ClosePrice"]
X = df.drop(columns=["ClosePrice", "HighSchoolDistrict", 'Unnamed: 0', "StoriesFinal"])

In [29]:
nan_percent = X.isna().mean() * 100
print(nan_percent)

ViewYN                   0.000000
PoolPrivateYN            0.000000
Latitude                 0.000000
Longitude                0.000000
LivingArea               0.011180
CountyOrParish           0.000000
AttachedGarageYN         0.000000
ParkingTotal             0.000000
BathroomsTotalInteger    0.000000
City                     0.000000
BedroomsTotal            0.000000
FireplaceYN              0.000000
LotSizeArea              2.037622
NewConstructionYN        0.000000
PostalCode               0.000000
BuildingAge              0.000000
dtype: float64


In [25]:
X

,ViewYN,PoolPrivateYN,Latitude,Longitude,LivingArea,CountyOrParish,AttachedGarageYN,ParkingTotal,BathroomsTotalInteger,City,BedroomsTotal,FireplaceYN,LotSizeArea,NewConstructionYN,PostalCode,BuildingAge
0,True,False,33.725080,-117.222302,2824.0,Riverside,True,2.0,3.0,Menifee,5.0,False,7000.0,True,92586,1.0
1,True,False,34.203479,-118.643567,2500.0,Los Angeles,True,2.0,3.0,Los Angeles,5.0,True,8336.0,False,91307,54.0
2,False,False,34.460368,-118.490755,2363.0,Los Angeles,True,2.0,3.0,Saugus,5.0,True,11705.0,False,91390,30.0
3,False,False,34.043218,-118.519477,3338.0,Los Angeles,False,2.0,5.0,Pacific Palisades,4.0,True,6502.0,False,90272,21.0
4,False,False,37.669495,-121.763793,2485.0,Alameda,True,2.0,3.0,Livermore,4.0,True,5093.0,False,94550,14.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35772,False,False,33.700193,-117.371513,1435.0,Riverside,True,2.0,2.0,Lake Elsinore,3.0,False,6680.0,True,92530,1.0
35773,True,False,34.378787,-117.330930,2226.0,San Bernardino,True,2.0,3.0,Hesperia,4.0,True,18000.0,False,92345,35.0
35774,True,False,34.854045,-119.148141,1753.0,Kern,True,2.0,3.0,Pine Mountain Club,3.0,True,10104.0,False,93225,19.0
35775,True,False,33.891018,-118.087100,1186.0,Los Angeles,False,4.0,1.0,Norwalk,2.0,True,6211.0,False,90650,76.0


In [11]:
# 3) Column types
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

In [13]:
# 4) Preprocessing
numeric_transformer = Pipeline([
    ("imputer", IterativeImputer(random_state=42, add_indicator=True))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

In [15]:
# 5) GB model
gb_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

In [17]:
# 6) Pipeline trained in log-space (log1p/expm1 is safer)
gb_log = Pipeline([
    ("preprocessor", preprocessor),
    ("model", gb_model)
])
ttr = TransformedTargetRegressor(
    regressor=gb_log,
    func=np.log1p,       # train on log1p(y)
    inverse_func=np.expm1  # predict back to raw scale
)

In [19]:
# 7) Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [31]:
# 8) Fit
ttr.fit(X_train, y_train)

TransformedTargetRegressor(func=<ufunc 'log1p'>, inverse_func=<ufunc 'expm1'>,
                           regressor=Pipeline(steps=[('preprocessor',
                                                      ColumnTransformer(transformers=[('num',
                                                                                       Pipeline(steps=[('imputer',
                                                                                                        IterativeImputer(add_indicator=True,
                                                                                                                         random_state=42))]),
                                                                                       ['Latitude',
                                                                                        'Longitude',
                                                                                        'LivingArea',
                                                                                        'ParkingTotal',
                                                                                        'BathroomsTotalInteger',
                                                                                        'BedroomsTotal',
                                                                                        'LotSizeArea',
                                                                                        'BuildingAge']),
                                                                                      ('cat',
                                                                                       Pipeline(steps=[('imputer',
                                                                                                        SimpleImputer(fill_value='Missing',
                                                                                                                      strategy='constant')),
                                                                                                       ('onehot',
                                                                                                        OneHotEncoder(handle_unknown='ignore'))]),
                                                                                       ['ViewYN',
                                                                                        'PoolPrivateYN',
                                                                                        'CountyOrParish',
                                                                                        'AttachedGarageYN',
                                                                                        'City',
                                                                                        'FireplaceYN',
                                                                                        'NewConstructionYN',
                                                                                        'PostalCode'])])),
                                                     ('model',
                                                      GradientBoostingRegressor(n_estimators=300,
                                                                                random_state=42))]))

In [ ]:
# 9) Predict (raw scale)
y_pred = ttr.predict(X_test)

In [22]:
# 10) Metrics
print("=== Gradient Boosting (trained on log-target) ===")
print("MAPE (raw):", mean_absolute_percentage_error(y_test, y_pred))
print("R² (raw):", r2_score(y_test, y_pred))

# Optional: R² in log-space (since training was in log-space)
y_test_log = np.log1p(y_test)
y_pred_log = np.log1p(y_pred)
print("R² (log1p):", r2_score(y_test_log, y_pred_log))

=== Gradient Boosting (trained on log-target) ===
MAPE (raw): 0.150919159944062
R² (raw): 0.7272112697715182
R² (log1p): 0.8766492357905009
